<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/Copy_of_CIFAR10_CIFAR100_TenSeed_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 and CIFAR-100 PEPITA: additional seeds

This notebook runs seeds 0–8 for both one-hidden-layer PEPITA + weight-decay experiments. Combined with the existing successful run, this gives ten observations per dataset. Existing baselines: CIFAR-10 seed 3868725372 (53.13%, CPU); CIFAR-100 seed 20261031 (24.53%, Tesla T4). The failed Blackwell attempt is excluded. CIFAR-10’s combined sample has one CPU result and nine GPU results; report this hardware difference.

Runs are sequential and may take several hours. Finish the currently running MNIST notebook first; do not share one GPU runtime between training notebooks. Completed run archives are saved to Drive after every seed, so rerunning this notebook skips finished seeds.


## 1. Select GPU, mount Drive, and upload source ZIP
Use T4 or faster. Upload the complete project source ZIP (`scripts/train.py` and `pepita/`), excluding `.venv-pepita`.

In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,shutil,statistics
from datetime import datetime,timezone
from google.colab import drive,files
print(sys.version)
!nvidia-smi
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/PEPITA_ten_seed_runs'); ROOT.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
assert len(uploaded)==1 and next(iter(uploaded)).lower().endswith('.zip')
ex=Path('/content/pepita_src'); shutil.rmtree(ex,ignore_errors=True); ex.mkdir()
with zipfile.ZipFile(next(iter(uploaded))) as z: z.extractall(ex)
ts=list(ex.rglob('scripts/train.py')); assert len(ts)==1, f'Expected one scripts/train.py; got {len(ts)}'
PROJECT=ts[0].parent.parent.resolve(); assert (PROJECT/'pepita/core/trainer.py').exists(); os.chdir(PROJECT)
print('Project:',PROJECT)


## 2. Install the pinned CUDA 11.7 / PyTorch 1.13 environment

In [ ]:
os.environ['MAMBA_ROOT_PREFIX']='/content/micromamba'
if not Path('/content/bin/micromamba').exists(): subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",shell=True,check=True)
MM='/content/bin/micromamba'
subprocess.run([MM,'create','-y','-n','pepita','python=3.10','pip'],check=True)
subprocess.run([MM,'run','-n','pepita','python','-m','pip','install','--upgrade','pip==24.0'],check=True)
pkgs=['torch==1.13.1+cu117','torchvision==0.14.1+cu117','pytorch-lightning==1.6.5','torchmetrics==0.9.2','numpy==1.21.2','Pillow==8.4.0','scikit-learn==1.2.1','scipy==1.10.1','loguru==0.6.0','setuptools==80.9.0','ray[tune]==2.3.1','yacs==0.1.8','tensorboard==2.12.3','protobuf<4']
subprocess.run([MM,'run','-n','pepita','python','-m','pip','install','--extra-index-url','https://download.pytorch.org/whl/cu117',*pkgs],check=True)


## 3. Apply two guarded compatibility fixes
The zero-split loader guard preserves all training data and uses the project’s test-loader validation hook. CIFAR-10 download is enabled only to fetch the canonical dataset; preprocessing is unchanged.

In [ ]:
patch=r'''from pathlib import Path
p=Path('pepita/dataset/getdataloader.py'); s=p.read_text(encoding='utf-8')
a='    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True)\n'
b='    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True) if val_data is not None else None\n'
assert s.count(a)==3, f'Expected 3 validation loader lines, found {s.count(a)}'
s=s.replace(a,b)
for old in ['datasets.CIFAR10(ds_directory, train=True, transform=trans_t, download=False)','datasets.CIFAR10(ds_directory, train=False, transform=trans, download=False)']:
 assert s.count(old)==1, f'Expected one: {old}'
 s=s.replace(old,old.replace('download=False','download=True'))
p.write_text(s,encoding='utf-8')
print('Compatibility/data-download patches applied.')
'''
subprocess.run([MM,'run','-n','pepita','python','-c',patch],cwd=PROJECT,check=True)
check="import torch; c=torch.cuda.get_device_capability(0); print(torch.__version__,torch.cuda.get_device_name(0),c); assert f'sm_{c[0]}{c[1]}' in torch.cuda.get_arch_list()"
subprocess.run([MM,'run','-n','pepita','python','-c',check],check=True)


## 4. Run the nine new seeds per dataset
Settings match the completed configurations: CIFAR-10 LR 0.01, WD 1e-4, WMLR 0.001, WMWD 0.1; CIFAR-100 LR 0.01, WD 1e-5, WMLR 0.1, WMWD 0.5. Both use 100 epochs, batch 64, momentum 0.9, dropout 0.1, and LR drops at epochs 60 and 90. Each run is saved to Drive immediately. Rerun this cell after reconnecting to continue.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import re
SEEDS=list(range(1,10))
CFG={'cifar100':dict(base_seed=20261031,base_acc=0.24529999494552612,base_loss=4.4813923835754395,base_hw='Tesla T4',wd=1e-5,wm_lr=0.1,wm_wd=0.5,workers=2),
     'cifar10':dict(base_seed=3868725372,base_acc=0.5313,base_loss=2.008038,base_hw='CPU (Windows)',wd=1e-4,wm_lr=0.001,wm_wd=0.1,workers=0)}
info_cmd="import torch,torchvision,pytorch_lightning as pl,platform,json; print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__,'lightning':pl.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}))"
runtime=json.loads(subprocess.check_output([MM,'run','-n','pepita','python','-c',info_cmd],cwd=PROJECT,text=True))
def extract_metrics(folder):
 ev=[p for p in folder.rglob('events.out.tfevents.*') if not any(x in str(p) for x in ('angle_al_total','weight_norms','b_means','b_stds'))]
 assert ev, f'No main TensorBoard file in {folder}'
 a=EventAccumulator(str(ev[0]),size_guidance={'scalars':0}); a.Reload(); ans={}
 for tag in ('val_acc','val_loss'):
  e=max(a.Scalars(tag),key=lambda e:e.step); ans[tag]=float(e.value); ans[tag+'_step']=int(e.step)
 return ans
for ds in ('cifar100','cifar10'):
 c=CFG[ds]; dest=ROOT/ds; dest.mkdir(parents=True,exist_ok=True)
 for seed in SEEDS:
  exp=f'{ds}_pepita_1hidden_seed{seed}'; saved=dest/f'{exp}.zip'
  if saved.exists(): print('Skip completed',exp); continue
  cmd=[MM,'run','-n','pepita','python','scripts/train.py','-en',exp,'-d',ds,'-ls','1024','-lr','0.01','-wd',str(c['wd']),'-e','100','-bs','64','-mom','0.9','-dr','0.1','-lrd','0.1','-de','60','90','-li','he_uniform','-bi','uniform','-bstd','0.05','-w',str(c['workers']),'-wmlr',str(c['wm_lr']),'-wmwd',str(c['wm_wd']),'-s',str(seed)]
  start=datetime.now(timezone.utc).isoformat(); print('START',exp,start,flush=True)
  folder=PROJECT/'experiments'/exp; shutil.rmtree(folder,ignore_errors=True); folder.mkdir(parents=True,exist_ok=True)
  with open(folder/'colab_console.log','w',encoding='utf-8') as log:
   p=subprocess.Popen(cmd,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
   for line in p.stdout: print(line,end='',flush=True); log.write(line)
   rc=p.wait()
  end=datetime.now(timezone.utc).isoformat()
  if rc: raise RuntimeError(f'{exp} failed ({rc}); earlier archives persist in Drive.')
  metrics=extract_metrics(folder)
  actual_seed=int(re.search(r'^SEED_VALUE:\s*(\d+)',(folder/'config.yaml').read_text(encoding='utf-8'),re.M).group(1))
  meta={'dataset':ds,'experiment':exp,'seed':actual_seed,'requested_seed':seed,'metrics':metrics,'runtime':runtime,'started_utc':start,'finished_utc':end,'note':'test loader monitored each epoch; final epoch only'}
  (folder/'final_metrics.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
  with zipfile.ZipFile(saved,'w',zipfile.ZIP_DEFLATED) as z:
   for f in folder.rglob('*'):
    if f.is_file(): z.write(f,Path('experiments')/exp/f.relative_to(folder))
  print('SAVED',saved,'metrics',metrics,flush=True)
print('All requested seeds completed or were already archived.')


## 5. Calculate statistics including the prior successful run
A dataset counts as ten seeds only when all nine new seeds completed and are archived. CIFAR-10 includes one prior CPU baseline; the mixed hardware is explicitly retained in the output.

In [ ]:
def read_zip_metrics(path):
 with zipfile.ZipFile(path) as z:
  n=next(n for n in z.namelist() if n.endswith('/final_metrics.json'))
  return json.loads(z.read(n).decode())
summary={}
for ds,c in CFG.items():
 rows=[{'seed':c['base_seed'],'accuracy':c['base_acc'],'loss':c['base_loss'],'hardware':c['base_hw'],'source':'prior successful run'}]
 for zp in sorted((ROOT/ds).glob('*.zip')):
  m=read_zip_metrics(zp); rows.append({'seed':m['seed'],'accuracy':m['metrics']['val_acc'],'loss':m['metrics']['val_loss'],'hardware':m['runtime']['gpu'],'source':zp.name})
 by_seed={}
 for r in rows:
  if r['seed'] not in by_seed or r['source']!='prior successful run': by_seed[r['seed']]=r
 rows=[by_seed[k] for k in sorted(by_seed)]; acc=[r['accuracy'] for r in rows]; loss=[r['loss'] for r in rows]
 summary[ds]={'n':len(rows),'runs':rows,'accuracy_mean':statistics.mean(acc),'accuracy_sample_sd':statistics.stdev(acc) if len(acc)>1 else None,'loss_mean':statistics.mean(loss),'loss_sample_sd':statistics.stdev(loss) if len(loss)>1 else None,'paper_mean':{'cifar10':0.5305,'cifar100':0.2486}[ds],'paper_sd':{'cifar10':0.0023,'cifar100':0.0018}[ds],'note':'CIFAR-10 includes one CPU run; remaining runs use GPU.' if ds=='cifar10' else 'All runs use Tesla T4.'}
print(json.dumps(summary,indent=2))
summary_path=ROOT/'ten_seed_summary.json'; summary_path.write_text(json.dumps(summary,indent=2),encoding='utf-8'); print('Saved:',summary_path)


## 6. Download the current summary
The per-run ZIP files remain in Google Drive and can be resumed later. Download this JSON summary to share the current seed counts and statistics.

In [ ]:
files.download(str(summary_path))